# Session 03: Attention and Hidden States

**Student implementer and maintainer:** YQ-HA. This notebook records code typed,
executed, and adapted while following the instructor during class, together with
the student's post-course completions and maintenance.

**Course:** Tyler Shoemaker, [Introduction to Interpretability for Language Models](https://digitaltheorylab.github.io/2026_interpretability-reader/).
Reference implementation: [03_attention.md](https://github.com/digitaltheorylab/2026_interpretability-reader/blob/main/chapters/03_attention.md).

**Repair assistance:** OpenAI Codex assisted with locating and explaining errors,
restoring omitted course code, and checking coverage and runnability. `Repair:`
comments identify corrected mistakes. Original classroom notes, including `##`
comments, are retained.

Run from the first cell with the kernel in `notebooks/`. New results go to
`results/session-03/`. This repaired version completed its Colab execution on October 26, 2026. See `results/` for the saved run; session 08 analyzed a truncated reasoning trace.


In [ ]:
from pathlib import Path
import json
import platform
import sys
import matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "session-03"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FACET_HEIGHT = 4
FACET_ASPECT = 1.2
def save_figure(name):
    plt.gcf().savefig(RESULTS_DIR / name, bbox_inches="tight", dpi=150)
    plt.show()
    plt.close("all")


In [ ]:
import math
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    attn_implementation="eager",
    low_cpu_mem_usage=True
)
model.eval()

embedding_layer = model.get_input_embeddings()
emb = embedding_layer.weight.detach().cpu().to(torch.float64)

plot_config = {
    "cmap": "Blues",
    "robust": True,
    "annot": True,
    "fmt": ".1f",
}
SQUARE_FIGSIZE = 6

In [ ]:
def attention(Q, K, V, mask=None):
    """
    scaling dot product attention
    Q, K, V: shape (n_tokens, d_model) or (n_tokens, d_k)
    mask:  (n_tokens, n_tokens)
    return: output (n_tokens, d_v), weights (n_tokens, n_tokens)
    """
    scores = Q @ K.T                     # (n, n)
    d_k = K.shape[1]
    scores = scores / math.sqrt(d_k)     # 
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)  # 
    output = weights @ V                 # 
    return output, weights

In [ ]:
text = "Then I tried to find some way of embracing my mother's ghost."
encoded = tokenizer.encode(text)
seq_emb = emb[encoded]   # torch.Tensor, shape (n_tokens, 768)

Q = K = V = seq_emb
output, weights = attention(Q, K, V)

print("Output shape:", output.shape)    # (13, 768)
print("Weights shape:", weights.shape)  # (13, 13)

# 
labels = [repr(tokenizer.decode(tok)) for tok in encoded]

# 
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(weights.detach().numpy(), **plot_config,
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title("Self-Attention Weights (No Mask)")
save_figure("s03-original-002.png")

In [ ]:
N = len(encoded)
mask = torch.tril(torch.ones(N, N))   # 

output, weights = attention(Q, K, V, mask=mask)

# 
zero_mask = (weights == 0).numpy()

fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(weights.detach().numpy(), mask=zero_mask, **plot_config,
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title("Causal Self-Attention (with Mask)")
save_figure("s03-original-003.png")

In [ ]:
# Repair: Seed randomized PCA; illustrative Q/K projections are distinct from learned model projections.
def project_embeddings(emb, seq_emb, n_components=64):
    pca = PCA(n_components=n_components, whiten=True, svd_solver="randomized", random_state=5167)
    pca.fit(emb.numpy())
    proj = pca.transform(seq_emb.numpy())
    return torch.tensor(proj, dtype=seq_emb.dtype, device=seq_emb.device)

seq_proj = project_embeddings(emb, seq_emb, n_components=64)
Q = K = seq_proj
V = seq_emb
output, weights = attention(Q, K, V, mask=mask)

print("Q/K shape:", Q.shape)   # (13, 64)
print("V shape:", V.shape)     # (13, 768)
print("Output shape:", output.shape)  # (13, 768)

In [ ]:
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

print("Number of layers:", len(outputs.attentions))  # 12
print("Attention tensor shape:", outputs.attentions[0].shape)  # (1, 12, 13, 13)

# average all heads to numpy
attentions = []
for tensor in outputs.attentions:
    # tensor: (1, 12, 13, 13) -> squeeze -> (12, 13, 13) -> mean over heads -> (13, 13)
    avg = tensor.squeeze(0).mean(dim=0).detach().cpu().numpy()
    attentions.append(avg)

# all heat map for all layers
fig, axes = plt.subplots(4, 3, figsize=(12, 16))
axes = axes.flatten()
for i, ax in enumerate(axes):
    sns.heatmap(attentions[i], mask=zero_mask, ax=ax, cmap="Blues",
                robust=True, cbar=False, xticklabels=labels, yticklabels=labels)
    ax.set_title(f"Layer {i}")
plt.suptitle("Layer-wise Attention Weights (Averaged over Heads)")
plt.tight_layout()
save_figure("s03-original-005.png")

In [ ]:
layer_2 = outputs.attentions[2].squeeze(0)   # (12, 13, 13)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
axes = axes.flatten()
for head_idx, ax in enumerate(axes):
    head_weights = layer_2[head_idx].detach().cpu().numpy()
    sns.heatmap(head_weights, mask=zero_mask, ax=ax, cmap="Blues",
                robust=True, cbar=False, xticklabels=labels, yticklabels=labels)
    ax.set_title(f"Head {head_idx}")
plt.suptitle("Attention Heads at Layer 2")
plt.tight_layout()
save_figure("s03-original-006.png")

In [ ]:
def strongest_previous(weights, labels, exclude_self=True, ignore=None):
    if ignore is None:
        ignore = []
    N = len(labels)
    for q in range(N):
        row = weights[q].copy()
        # MASK
        for k in range(q+1, N):
            row[k] = -1
        if exclude_self:
            row[q] = -1
        for k in ignore:
            row[k] = -1
        if row.max() < 0:
            continue
        k = row.argmax()
        print(f"{labels[q]:<15} --> {labels[k]:<15} {row[k]:.3f}")

# measure avg w layer 2.
print("Layer 2 averaged:")
strongest_previous(attentions[2], labels)

# test layer 2 head 3.
print("\nLayer 2, Head 3:")
strongest_previous(layer_2[3].detach().cpu().numpy(), labels)

# ignore then.
print("\nLayer 0 ignoring 'Then':")
strongest_previous(attentions[0], labels, ignore=[0])

In [ ]:
# Repair: Clarify the original state slice while keeping the classroom note.
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

hidden = torch.stack(outputs.hidden_states).squeeze(1).detach().cpu()  # (13, 13, 768)

# first 12 layer without LayerNorm.
# Clarification: hidden[:-1] retains embeddings and excludes the final normalized output.
traj = hidden[:-1]   # (12, 13, 768)
n_layers, n_tokens, dim = traj.shape

# standardize
flat = traj.reshape(-1, dim).numpy()
scaler = StandardScaler()
scaled = scaler.fit_transform(flat)

# PCA to 2 dim
pca = PCA(n_components=2)
proj = pca.fit_transform(scaled).reshape(n_layers, n_tokens, 2)

# token trace.
selected = [8, 9, 10, 11]   # "embracing", "my", "mother's", "ghost."
fig, ax = plt.subplots(figsize=(SQUARE_FIGSIZE, SQUARE_FIGSIZE))
for idx in selected:
    path = proj[:, idx, :]   # (12, 2)
    ax.plot(path[:, 0], path[:, 1], marker='o', label=labels[idx])
    # 0,6,11 layer.
    for layer_idx in [0, n_layers//2, n_layers-1]:
        ax.text(path[layer_idx, 0], path[layer_idx, 1], str(layer_idx),
                ha='center', va='center', fontsize=8)
ax.legend()
ax.set_title("Token Trajectories through Layers (PCA)")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
save_figure("s03-original-008.png")

In [ ]:
import numpy as np
distances = np.linalg.norm(np.diff(proj, axis=0), axis=-1).sum(axis=0)
fig, ax = plt.subplots()
sns.barplot(x=labels, y=distances, ax=ax)
ax.tick_params(axis="x", rotation=90)
ax.set(title="Token movement in the PCA projection", ylabel="Path length in projected space")
save_figure("s03-course-completion-001.png")


In [ ]:
observed = {name: module.__version__ for name, module in sys.modules.items()
            if name in {"numpy", "pandas", "torch", "transformers", "sklearn", "scipy", "matplotlib", "seaborn"}}
environment = {"python": platform.python_version(), "imported_packages": observed}
(RESULTS_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Completed notebook; results:", RESULTS_DIR)
